In [0]:
# ================================================
# READ FROM CLEANING OUTPUT
# ================================================
df_final_clean = spark.table("default.nottinghamshire_clean")

print(f"✅ Rows loaded: {df_final_clean.count()}")
print(f"✅ Columns: {df_final_clean.columns}")

In [0]:
# ================================================
# VALIDATION LAYER
# ================================================
from pyspark.sql.functions import col, count, when

print("=== NULL CHECK ON KEY REPORTING FIELDS ===")
key_fields = [
    "force_name", "year", "quarter",
    "month_num", "month_name", "crime_type"
]
for field in key_fields:
    null_count = df_final_clean.filter(
        col(field).isNull()
    ).count()
    status = "✅" if null_count == 0 else "⚠️"
    print(f"{status} {field}: {null_count} nulls")

print("\n=== YEAR RANGE CHECK ===")
df_final_clean.groupBy("year") \
    .count() \
    .orderBy("year") \
    .show()

print("\n=== MONTH COVERAGE CHECK ===")
df_final_clean.groupBy("year", "month_num") \
    .count() \
    .orderBy("year", "month_num") \
    .show(100)

print("\n=== CRIME TYPE CHECK ===")
df_final_clean.groupBy("crime_type") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

print("\n=== QUARTER CHECK ===")
df_final_clean.groupBy("quarter") \
    .count() \
    .orderBy("quarter") \
    .show()

In [0]:
# Check which non-Nottinghamshire areas appear
from pyspark.sql.functions import substring

df_final_clean.filter(
    ~col("lsoa_name").rlike(
        "Nottingham|Ashfield|Mansfield|Newark|Bassetlaw|Broxtowe|Gedling|Rushcliffe|Amber Valley|Erewash"
    )
) \
.groupBy("lsoa_name") \
.count() \
.orderBy("count", ascending=False) \
.show(50, truncate=False)

In [0]:
# ================================================
# SAVE VALIDATED DATA
# So aggregation notebook can read it
# ================================================
df_final_clean.write \
    .mode("overwrite") \
    .saveAsTable("default.nottinghamshire_validated")

print("✅ Validated data saved to default.nottinghamshire_validated")
print(f"✅ Total rows saved: {df_final_clean.count()}")

In [0]:
df_clean_check = spark.table("default.nottinghamshire_clean")
print(f"Clean table rows: {df_clean_check.count():,}")
df_clean_check.groupBy("year").count().orderBy("year").show()

In [0]:
df_validated_check = spark.table("default.nottinghamshire_validated")
print(f"Validated table rows: {df_validated_check.count():,}")
df_validated_check.groupBy("year").count().orderBy("year").show()